In [ ]:
srt(secure reliable transport) : udp (we can use retry mechanism here) its support h264, h265 etc, so comprassion can reduce
rtmp : tcp only support h264 , persistance connection, low latency


In [ ]:
JD: 

Project Description
This project focuses on developing a robot/drone control service that utilizes DeHive’s own robots and drones for
safety monitoring purposes.
The goal is to provide services to multiple clients, enabling client-specific / region-specific / robot-specific control and
AI-based real-time recognition and recording of video feeds transmitted from robots. The stored data should be
retrievable by client, site, or robot for later review.
Key development areas include:
❖ Full-stack server architecture and cloud data management
❖ Real-time video transmission, playback, and AI-based recognition
❖ User, client, and robot device management
❖ Visualization and replay of stored video, GPS, and Lidar sensor data
EXPERIENCE
Duties you'll be entrusted with:
❖ Software Developer (Full Stack) responsible for cloud-related POC services.
Expectations from you:
Mandatory Requirements
❖ Video Streaming (RTMP, WebRTC)
❖ Vision AI (ML, python)
❖ Front end (React.js)
❖ Database (DynamoDB or PostgreSQL)
❖ Messaging(MQTT, socket.io)
❖ AWS
❖ Docker
Additional (Optional) Requirement
❖ Back-end (PHP, Laravel, FastAPI)
❖ Experience in developing AI or robotics-related services
❖ WebApp, C++, Figma
❖ Cloud Service full stack architect, Security, Fleet management, UI/UX Design, LLM

## streamyard: its a browser based video streaming platform. 

my camera feed to be stream over any browser this is called RTMP

In [ ]:
https://www.youtube.com/watch?v=JwZiO5p-NAE

In [ ]:
RTMP based on TCP : more reliable

create hls 
https://www.youtube.com/watch?v=6JTV4PwisoQ 

In [ ]:
1. video upload on s3 
2. in docker container we can process this video 
3. converting video into chunks and segments
4. with help of ffmpeg


## creating hls segment using ffmpeg for recorded video

In [ ]:
## ffmpeg command  for creating hls chunks for recorded video
ffmpeg -i dj_wale_babu.mp4 -codec:v libx264 -codec:a aac -hls_time 10 -hls_playlist_type vod -hls_segment_filename hls_output/segment%03d.ts -start_number 0 hls_output/index.m3u8

In [ ]:
## ffmppeg for playing m3u8 file
ffplay hls_output/index.m3u8


/bin/bash: line 1: gnome: command not found


## for sending live stream on rtmp link for recorded video

In [ ]:
ffmpeg -re -i dj_wale_babu.mp4 \
-c:v libx264 -preset veryfast -tune zerolatancy \
-c:a aac --44100 -b:a 128k \
-f flv rtmp://127.0.0.1/live/test

## for live streaming for webcam

In [ ]:
## for recorded video
ffmpeg -i dj_wale_babu.mp4 \
-codec:v libx264 -codec:a aac \
-hls_time 10 -hls_playlist_type vod \
-hls_segment_filename hls_output/segment%03d.ts \
-start_number 0 hls_output/index.m3u8


## for only video
ffmpeg -f v4l2 -i /dev/video0 \
-c:v libx264 -preset veryfast -tune zerolatency \
-f flv rtmp://127.0.0.1/live/webcam

## for video and audio
ffmpeg -f v4l2 -i /dev/video0 \
-c:v libx264 -present veryfast -tune zerolatency \
-c:a aac -ar 44100 -b:a 128k \
-f flv rtmp://127.0.0.1/live/webcam

## for fps and resolution
ffmpeg -f v4l2 -framerate 30 -video_size 1280x720 -i /dev/video0 \
-f alsa -i hw:1,0 \
-c:v libx264 -preset veryfast -tune zerolatency \
-pix_fmt yuv420p \
-c:a aac -ar 44100 -b:a 128k \
-f flv rtmp://127.0.0.1/live/webcam


## at client side

ffplay rtmp://127.0.0.1/live/webcam


## input devices video and audio 
v4l2-clt --list-devices
arecord -l


In [ ]:
🏗 Architecture Explanation (End-to-End Flow)
High-Level Architecture
TV / Mobile Device
        ↓
     Webcam
        ↓
     FFmpeg
   (Capture + Encode)
        ↓
 ┌───────────────┬────────────────┐
 │               │                │
RTMP Server      HLS Segmenter     Local Recording
(Live View)   (.ts + .m3u8)        (Optional)
 │               │
Live Clients     S3 Bucket
                  │
            VOD Playback

Step-by-Step Explanation

Live Source (TV & Mobile)

The Euro Cup / IPL live broadcast was played on real TV and mobile devices

Devices were placed in front of a webcam to capture exactly what an end user sees

Capture & Encoding (FFmpeg)

FFmpeg captured:

Video from the webcam (V4L2)

Audio from a microphone (ALSA)

Encoded video using H.264 (libx264) and audio using AAC

Optimized encoding for low latency

Live Streaming (RTMP)

The encoded stream was pushed to an RTMP server

Clients could watch the stream in near real-time

Recording & HLS Generation

The same FFmpeg pipeline generated:

.ts video segments

.m3u8 playlist file

This converted the live stream into HLS format

Storage & Playback (S3 + HLS)

HLS files were uploaded to an S3 bucket

Clients could access recordings anytime using the HLS (.m3u8) URL

This enabled Video-on-Demand (VOD) playback

⚙ Challenges Faced & How I Optimized Them
1️⃣ Latency Issues (Live Streaming)

Challenge

RTMP + encoding introduces delay

Webcam capture and buffering increase latency

Optimizations

Used:

-preset veryfast -tune zerolatency


Reduced internal buffering:

-fflags nobuffer -flags low_delay


Tuned GOP size to reduce startup delay

Avoided unnecessary re-encoding

Result

Achieved near real-time streaming suitable for monitoring live matches

2️⃣ Audio–Video Sync Problems

Challenge

Audio and video came from separate devices

Drift occurred during long streams

Optimizations

Explicitly defined:

Frame rate

Audio sample rate

Used FFmpeg’s sync handling:

-vsync 1 -async 1


Ensured consistent timestamps

Result

Stable audio-video sync throughout long match recordings

3️⃣ Storage Growth & File Management

Challenge

Long matches (3–4 hours) generated:

Thousands of HLS segments

Large storage usage on S3

Optimizations

Configured:

Smaller segment duration (balanced latency & size)

Automatic cleanup for older segments

Used S3 lifecycle policies:

Move old recordings to Glacier

Delete after defined retention period

Result

Controlled storage cost without affecting playback

4️⃣ Reliability & Recovery

Challenge

Network issues could interrupt live streaming

FFmpeg process crashes would stop recording

Optimizations

Wrapped FFmpeg in:

Supervisor / systemd

Auto-restart logic

Used segmented HLS so partial failures didn’t lose full recordings

Result

High reliability during long live events

5️⃣ Playback Compatibility

Challenge

Playback needed to work across:

Web

Mobile

Different players

Optimizations

Used:

-pix_fmt yuv420p


Standard H.264 + AAC codecs

Ensured HLS compliance

Result

Smooth playback across all major devices and players

🎯 How This Sounds to an Interviewer

✔ Real-world problem
✔ Clear architecture
✔ Trade-offs explained
✔ Performance & cost optimizations
✔ Production mindset

🧾 Optional Resume Bullet (If You Want)

Designed and implemented a live streaming and recording pipeline using FFmpeg, RTMP, and HLS to capture real-device broadcasts (TV and mobile) for major sports events. Optimized for low latency, A/V sync, and scalable storage using AWS S3.

If you want, I can also:

Draw a simple architecture diagram explanation

Add numbers (latency %, storage reduction)

Prepare follow-up Q&A interviewers usually ask on this topic

Just say the word 👍

Prepare follow-up Q&A interviewers usually ask on this topic

Great idea — interviewers almost always go deeper on this topic.
Below is a ready-to-use follow-up Q&A set with clear, confident answers you can give verbatim or adapt.

🎤 Common Follow-Up Interview Q&A

(Live Streaming | FFmpeg | RTMP | HLS)

1️⃣ Why did you use RTMP for live and HLS for recording?

Answer:

RTMP provides lower latency and is well-suited for live monitoring, while HLS is more reliable 
for recording and playback. HLS also works across browsers and mobile devices and integrates 
well with S3. Using both allowed us to optimize for real-time viewing and long-term access.

2️⃣ Why didn’t you stream live using HLS directly?

Answer:

HLS introduces higher latency because it relies on segmented files and playlists. For live 
monitoring, that delay was not acceptable. RTMP allowed near real-time viewing, while HLS 
was generated in parallel for recordings.

3️⃣ How did you handle audio–video sync issues?

Answer:

Audio and video came from different devices, so I explicitly controlled frame rate and audio 
sample rate in FFmpeg and enabled FFmpeg’s sync mechanisms. I also monitored drift during 
long streams and adjusted buffering to keep timestamps aligned.

4️⃣ How did you reduce live streaming latency?

Answer:

I optimized encoding using veryfast preset and zerolatency tuning, reduced internal buffering, 
and avoided unnecessary re-encoding. I also tuned GOP size to reduce delay without impacting 
stream stability.

5️⃣ What happens if FFmpeg crashes during a live match?

Answer:

FFmpeg was wrapped with a process manager like systemd or supervisor. Since HLS works in 
segments, already generated files remained usable, and the process could resume without losing 
the entire recording.

6️⃣ How did you manage large storage usage on S3?

Answer:

Long matches generate many HLS segments, so we used S3 lifecycle policies to move older 
recordings to cheaper storage like Glacier or delete them after a retention period. This helped
control costs without affecting access.

7️⃣ Why did you choose H.264 and AAC codecs?

Answer:

H.264 and AAC offer the best balance of compression, quality, and compatibility. They are 
universally supported across browsers, mobile devices, and HLS players, which was important 
for client access.

8️⃣ How scalable is this solution?

Answer:

The solution is horizontally scalable. Multiple FFmpeg pipelines can run in parallel for 
different devices or events. Since RTMP and HLS are decoupled, viewers don’t directly impact 
ingestion performance.

9️⃣ Could this be extended to cloud-based streaming?

Answer:

Yes. The RTMP endpoint can be moved to cloud services, and HLS output can be pushed to S3 or 
a CDN. This would allow global access and better reliability without changing the core pipeline.

🔟 Why not use WebRTC for live streaming?

Answer:

WebRTC offers ultra-low latency but is more complex to manage at scale and not ideal for 
recording or long-duration streams. RTMP + HLS provided a simpler, more reliable solution for 
this use case.

1️⃣1️⃣ How did you ensure playback compatibility across devices?

Answer:

I used standard codecs, compatible pixel formats, and compliant HLS settings. We tested playback
on web browsers and mobile devices to ensure consistent behavior.

1️⃣2️⃣ What were the biggest technical risks in this project?

Answer:

The main risks were latency, audio-video sync drift, and long-duration stability. These were 
mitigated through encoding optimizations, explicit sync handling, and monitoring FFmpeg processes 
during events.

1️⃣3️⃣ If you had more time, what would you improve?

Answer:

I would add monitoring and alerting for stream health, automate failure recovery further, and 
introduce a CDN in front of S3 for faster playback across regions.

1️⃣4️⃣ How did you validate stream quality?

Answer:

We monitored live RTMP streams, reviewed recorded HLS playback, and checked for frame drops, sync 
issues, and visual clarity over long durations.

1️⃣5️⃣ Can you summarize the solution in one sentence?

Answer:

We built a low-latency live streaming and recording pipeline using FFmpeg, RTMP, and HLS to capture 
real-device broadcasts and make them available both live and on-demand.

In [ ]:
## using subprocess THIS IS blocking command
import subprocess
command = ["ls", "-l"]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

total 257476
-rw-rw-r-- 1 ashwin ashwin  11209324 May 28  2022 dj_wale_babu.mp4
-rw-rw-r-- 1 ashwin ashwin     12053 Jan  3 19:17 gstreamer.ipynb
drwxrwxr-x 2 ashwin ashwin      4096 Jan  5 13:18 hls_output
-rw-rw-r-- 1 ashwin ashwin 252373734 Nov 15 12:41 KT_2025_FastAPI_Docker.mp4
-rw-rw-r-- 1 ashwin ashwin     17408 Jan  5 15:16 rtmp.ipynb
-rw-rw-r-- 1 ashwin ashwin      4648 Dec  9 12:25 webrtc.ipynb
-rw-rw-r-- 1 ashwin ashwin       982 Oct  5 23:34 websocket.ipynb
-rw-rw-r-- 1 ashwin ashwin       392 Oct  5 23:33 wsclinet.py
-rw-rw-r-- 1 ashwin ashwin       795 Dec  6 15:57 wsserver.py




In [2]:
## run using shell

subprocess.run(command, shell=True)

dj_wale_babu.mp4
gstreamer.ipynb
hls_output
KT_2025_FastAPI_Docker.mp4
rtmp.ipynb
webrtc.ipynb
websocket.ipynb
wsclinet.py
wsserver.py


CompletedProcess(args=['ls', '-l'], returncode=0)

In [ ]:
## for non blocking long running command
import subprocess
cmd = []
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)


## read live logs

import subprocess
process=subprocess.Popen(cmd, stderr=subprocess.PIPE, text=True)
for line in process.stderr:
    print(line.strip())


## for terminate and kill
process.terminate()
process.kill()

## check status
process.poll()

    